# Colab Setup Test
Minimal notebook to confirm: clone → data → import src → EDA works.
Run top to bottom on a fresh Colab session.

In [1]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("/content/DL_Proj"):
        !git clone https://github.com/fmssilva/DL_Proj.git /content/DL_Proj
    os.chdir("/content/DL_Proj/assignment_1")
    !pip install -r requirements.txt -q   # installs deps only — no package registration

# always insert assignment_1/ at the front of sys.path so 'import src.*' resolves
# from the filesystem directly, regardless of what pip has or hasn't installed
ROOT = os.getcwd()  # should be .../assignment_1
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# quick sanity check — print the first few entries so we can see where Python looks
print("cwd     :", ROOT)
print("sys.path:", sys.path[:3])



# ── Cell 2: Data ─────────────────────────────────────────────────────────────
# Locally:  data/ already on disk — skipped automatically.
# Colab:    downloads zip from public Google Drive link, extracts to data/.
#           No secrets, no file upload, no Kaggle account needed.
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

_GDRIVE_FILE_ID = "1nVSQZxQubLEPXjSRqGn7rtPzkw-S0zIi"

if not Path("data/train_labels.csv").exists():
    if IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "gdown", "-q"], check=True)
        import gdown
        gdown.download(id=_GDRIVE_FILE_ID, output="data.zip", quiet=False)
        with zipfile.ZipFile("data.zip") as zf:
            zf.extractall("data")
        os.remove("data.zip")
        print("data/ ready:", os.listdir("data"))
    else:
        print("ERROR: data/ not found. Make sure data/ is in assignment_1/.")
else:
    print("data/ already present — skipping download.")


# ── Cell 3: Verify everything works ──────────────────────────────────────────
import torch
import pandas as pd
from src.config import SEED, CLASSES, DATA_DIR, create_output_dirs, set_seed

set_seed(SEED)
create_output_dirs()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
df = pd.read_csv(DATA_DIR / "train_labels.csv")

print(f"Device  : {device}")
print(f"Classes : {CLASSES}")
print(f"Rows    : {len(df)}")
print("All good — src imports work.")



# ── Cell 3b: Import diagnostics ──────────────────────────────────────────────
# Run this if any import fails — shows exactly what Python finds and where.
# All imports inside src/ use relative imports (from ..config) so they work
# regardless of how sys.path is set up.
import sys, importlib

print("--- sys.path (first 5) ---")
for p in sys.path[:5]:
    print(" ", p)

print("\n--- src package location ---")
import src
print("  src:", src.__file__)

print("\n--- sub-package probes ---")
for mod in [
    "src.config",
    "src.data",
    "src.data.dataset",
    "src.data.eda",
    "src.data.eda_plots",
    "src.training.early_stopping",
    "src.training.train",
    "src.evaluation.metrics",
    "src.evaluation.submission",
    "src.evaluation.plots",
    "src.models.mlp",
]:
    try:
        m = importlib.import_module(mod)
        print(f"  OK   {mod:45s} {m.__file__}")
    except Exception as e:
        print(f"  FAIL {mod:45s} {e}")



# ── Cell 4: Quick EDA smoke test ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import src.data.eda as eda
import src.data.eda_plots as eda_plots
from src.config import DATA_DIR

TRAIN_DIR = DATA_DIR / "Train"

print("=== Class Distribution ===")
eda.class_distribution(df)


print("EDA works.")

cwd     : c:\Users\franc\OneDrive\Nossa_Pasta_2\5. Universidade\Cadeiras\DL\DL_Proj\assignment_1
sys.path: ['c:\\Users\\franc\\OneDrive\\Nossa_Pasta_2\\5. Universidade\\Cadeiras\\DL\\DL_Proj\\assignment_1', 'c:\\Users\\franc\\anaconda3\\envs\\cnn\\python310.zip', 'c:\\Users\\franc\\anaconda3\\envs\\cnn\\DLLs']
data/ already present — skipping download.
Device  : cpu
Classes : ['Bug', 'Fighting', 'Fire', 'Grass', 'Ground', 'Normal', 'Poison', 'Rock', 'Water']
Rows    : 3600
All good — src imports work.
--- sys.path (first 5) ---
  c:\Users\franc\OneDrive\Nossa_Pasta_2\5. Universidade\Cadeiras\DL\DL_Proj\assignment_1
  c:\Users\franc\anaconda3\envs\cnn\python310.zip
  c:\Users\franc\anaconda3\envs\cnn\DLLs
  c:\Users\franc\anaconda3\envs\cnn\lib
  c:\Users\franc\anaconda3\envs\cnn

--- src package location ---
  src: c:\Users\franc\OneDrive\Nossa_Pasta_2\5. Universidade\Cadeiras\DL\DL_Proj\assignment_1\src\__init__.py

--- sub-package probes ---
  OK   src.config                         